### Chain of thoughts prompting

Tests to see if we can manage some sort of chain of thought prompting. The output might need to be rethought in order to be parsable (pass through another LLM ?).

#### Naïve (hard-coded) version

In [1]:
# Install libraries

!pip install openai
!pip install openrouter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [openai]2m5/6 [openai]


In [4]:
# Define openrouter api key in a .env file

from dotenv import load_dotenv
import os

load_dotenv()  # loads .env into environment
token = os.environ["OPENROUTER_API_KEY"]


In [5]:
# Chatbot connection

from openrouter import OpenRouter
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_base = "https://openrouter.ai/api/v1/chat/completions"
model = "openai/gpt-oss-120b:free"
openai_api_key = "Bearer " + token
 
client = OpenRouter(
    api_key=openai_api_key
)

In [6]:
original_intro_prompt = """You are an helpful AI assistant who writes SQL query for a given question. Given the database described by the database schema below, write a SQL query that answers the question.\n
Do not explain the SQL query.\n
Return just the query, so it can be run verbatim from your response.\n"""

database_info_examples_and_questions = """### Database Schema\n

### Database Schema

CREATE TABLE experiment.spatial_unit (
    spatialunit_uid VARCHAR(30) NOT NULL,
    spatialunit_current_id INTEGER NOT NULL,
    spatialunit_chology VARCHAR(30) NOT NULL,
    name VARCHAR(100) NOT NULL,
    name_de VARCHAR(100),
    name_fr VARCHAR(100),
    name_it VARCHAR(100),
    country BOOLEAN DEFAULT false NOT NULL,
    canton BOOLEAN DEFAULT false NOT NULL,
    district BOOLEAN DEFAULT false NOT NULL,
    municipal BOOLEAN DEFAULT false NOT NULL,
    resise_area BOOLEAN DEFAULT false NOT NULL,
    neighborhood BOOLEAN DEFAULT false NOT NULL,
    region BOOLEAN DEFAULT false NOT NULL,
    zone BOOLEAN,
    spatialunit_hist_id INTEGER,
    canton_hist_id INTEGER,
    district_hist_id INTEGER,
    valid_from DATE,
    valid_until DATE,
    CONSTRAINT spatial_unit_pkey PRIMARY KEY (spatialunit_uid)
)

/*
Columns in spatial_unit and 5 examples in each column for high cardinality columns :
spatialunit_uid : 6195_A.ADM3, 695_A.ADM3, 6152_A.ADM3, 2783_A.ADM3, 4065_A.ADM3
spatialunit_current_id : 2092, 2304, 6195, 5169, 4311
name : Berlens, Uebeschi, Saint-Gingolph, Jongny, Hallwilersee (LU)
name_de : Berlens, Uebeschi, Saint-Gingolph, Jongny, Hallwilersee (LU)
name_fr : Berlens, Uebeschi, Saint-Gingolph, Jongny, Hallwilersee (LU)
name_it : Berlens, Uebeschi, Saint-Gingolph, Jongny, Hallwilersee (LU)
spatialunit_hist_id : 15230, 13245, 14740, 11434, 10088
canton_hist_id : 16, 14, 10, 23, 1
district_hist_id : 10266, 10229, 10009, 10313, 10088
valid_from : 1995-01-01, 2001-04-13, 2018-01-01, 2011-01-01, 2017-01-01
valid_until : 2010-04-24, 2017-12-31, 2012-12-31, 1967-05-31, 1971-12-31
*/

CREATE TABLE experiment.stock_vehicles (
    uid BIGINT NOT NULL,
    spatialunit_uid VARCHAR(100),
    vehicle_type VARCHAR(50),
    year INTEGER,
    amount INTEGER,
    fuel_type VARCHAR(50),
    CONSTRAINT stock_vehicles_pk PRIMARY KEY (uid),
    CONSTRAINT stock_vehicles_spatial_unit_uid_fk FOREIGN
        KEY(spatialunit_uid) REFERENCES experiment.spatial_unit (spatialunit_uid)
)

/*
Columns in stock_vehicles and 5 examples in each column for high cardinality columns :
uid : 1601120, 1601106, 1601202, 1601155, 1601156
spatialunit_uid : 5307_A.ADM3, 5430_A.ADM3, 3237_A.ADM3, 5912_A.ADM3, 5926_A.ADM3
year : 2015, 2019, 2020, 2010, 2012
amount : 16, 41, 305, 877, 1599
fuel_type : Diesel-electric: Plug-in-hybrid, Hydrogen, total, Benzine-electric: Plug-in-hybrid, Diesel
*/

/*
Column name, Column description, Example values
vehicle_type, Vehicle type, passenger_cars, trailers, motorcycles
amount, Amount of vehicles, 0, 2, 240
fuel_type, Fuel type, Hydrogen, Electric, Diesel-electric: Normal-hybrid
*/

### Example question
What is the proportion of electric vehicles of each type in the most recent year, in descending order?

### SQL query
SELECT
    vehicle_type,
    SUM(CASE WHEN fuel_type ILIKE '%electric%' THEN amount ELSE 0 END) AS electric_cars,
    SUM(amount) AS total_cars,
    (
        SUM(CASE WHEN fuel_type ILIKE '%electric%' THEN amount ELSE 0 END) * 100.0 /
        NULLIF(SUM(amount), 0)
    ) AS proportion_electric
FROM stock_vehicles
WHERE year = (
    SELECT MAX(year)
    FROM stock_vehicles
)
GROUP BY vehicle_type
ORDER BY proportion_electric DESC;


### Example question
Which cities do not have any electric cars registered?

### SQL query
SELECT su.name AS city
FROM spatial_unit AS su
WHERE NOT EXISTS (
    SELECT 1
    FROM stock_vehicles AS sv
    WHERE su.spatialunit_uid = sv.spatialunit_uid
      AND sv.fuel_type = 'Electric'
);


### Real question
What was the proportion of electric vehicles in Geneva in 2010?
"""

chat_response = client.chat.send(
    model=model,
    messages=[
        {"role": "system", "content": original_intro_prompt},
        {"role": "user", "content": database_info_examples_and_questions},
    ],
    provider={
        "zdr": False
    }
)
print("Chat response:")
print(chat_response.model_dump()["choices"][0]["message"]["content"])

Chat response:
SELECT
    SUM(CASE WHEN sv.fuel_type ILIKE '%electric%' THEN sv.amount ELSE 0 END) * 100.0 /
    NULLIF(SUM(sv.amount), 0) AS proportion_electric_percent
FROM experiment.spatial_unit su
JOIN experiment.stock_vehicles sv
  ON su.spatialunit_uid = sv.spatialunit_uid
WHERE su.name = 'Geneva'
  AND sv.year = 2010;


In [7]:
cot_intro_prompt = """You are an helpful AI assistant who writes SQL query for a given question. You have a description of the database schema below as well as a few examples of questions and SQL queries answers.\n
Write a SQL query that answers the last question. \n
Decompose and explain your thought process step by step.\n
Finish your answer with the whole SQL so it can be run verbatim.\n"""

chat_response = client.chat.send(
    model=model,
    messages=[
        {"role": "system", "content": cot_intro_prompt},
        {"role": "user", "content": database_info_examples_and_questions},
    ]
)
print("Chat response:")
print(chat_response.model_dump()["choices"][0]["message"]["content"])

Chat response:
**Step‑by‑step reasoning**

1. **Identify the spatial unit that corresponds to Geneva**  
   The table `experiment.spatial_unit` contains the column `name` (and its translations).  
   We filter rows where `name = 'Geneva'` (or the French/Italian equivalents if needed).  

2. **Link the vehicles to that spatial unit**  
   `experiment.stock_vehicles` stores a `spatialunit_uid` that references `spatial_unit.spatialunit_uid`.  
   By joining on this key we obtain all vehicle records that belong to Geneva.

3. **Restrict to the year 2010**  
   Add a condition `sv.year = 2010`.

4. **Calculate total vehicles and electric vehicles**  
   * Total = `SUM(sv.amount)`  
   * Electric = `SUM(CASE WHEN sv.fuel_type ILIKE '%electric%' THEN sv.amount ELSE 0 END)`

5. **Compute the proportion**  
   `electric * 100.0 / NULLIF(total,0)` gives the percentage; `NULLIF` avoids division by zero.

6. **Return a single row with the proportion (and optionally the raw counts)**  

**Final SQL

In [ ]:
!pip install psycopg2

In [13]:
# Enter database connection info in .env
import psycopg2 as pgr
import os

from dotenv import load_dotenv
load_dotenv()  # loads .env into environment

dbconn = pgr.connect(dbname=os.environ["DB_DATABASE"],user=os.environ["DB_USERNAME"],password=os.environ["DB_PASS"],host=os.environ["DB_HOST"],port=os.environ["DB_PORT"],options="-c search_path="+os.environ["DB_SCHEMA"])
dbconn.set_session(readonly=True)

In [20]:
qry = """
SELECT 
    su.name AS city,
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) AS electric_cars,
    COUNT(sv.uid) AS total_cars,
    (
        COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) * 100.0 /
        NULLIF(COUNT(sv.uid), 0)
    ) AS proportion_electric
FROM 
    spatial_unit AS su
JOIN 
    stock_vehicles AS sv ON su.spatialunit_uid = sv.spatialunit_uid
WHERE 
    sv.year = 2010
    AND su.name = 'Geneva'
GROUP BY 
    su.name
HAVING 
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) > 0
ORDER BY 
    proportion_electric DESC;
"""

with dbconn:
    with dbconn.cursor() as curs:
        try:
            curs.execute(qry)
            result = curs.fetchall()
            print(result)
            res = [rw for rw in result]
            colnames = [desc.name for desc in curs.description]
            if res and colnames:
                print(res)
                print(colnames)
            else:
                print(colnames)
        except Exception as e:
            print(e)

[]
['city', 'electric_cars', 'total_cars', 'proportion_electric']


In [ ]:
# Correction
answer = """
### Decomposition and Explanation

To answer the question, we need to perform the following steps:

1.  Get the most recent year from the `stock_vehicles` table.
2.  Filter the `stock_vehicles` table to include only the vehicles from the most recent year.
3.  Filter the `stock_vehicles` table to include only the electric vehicles.
4.  Join the `stock_vehicles` table with the `spatial_unit` table to get the city names.
5.  Calculate the proportion of electric vehicles for each city.

### SQL Query

```sql
SELECT 
    su.name AS city,
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) AS electric_cars,
    COUNT(sv.uid) AS total_cars,
    (
        COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) * 100.0 /
        NULLIF(COUNT(sv.uid), 0)
    ) AS proportion_electric
FROM 
    spatial_unit AS su
JOIN 
    stock_vehicles AS sv ON su.spatialunit_uid = sv.spatialunit_uid
WHERE 
    sv.year = (
        SELECT MAX(year)
        FROM stock_vehicles
    )
GROUP BY 
    su.name
HAVING 
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) = 0
ORDER BY 
    proportion_electric DESC;
```

However, the above query may not give the desired result as it only returns cities with no electric cars. To get the proportion of electric vehicles in Geneva in 2010, we need to filter the `stock_vehicles` table to include only the vehicles from Geneva in 2010 and then calculate the proportion of electric vehicles.

### Corrected SQL Query

```sql
SELECT 
    su.name AS city,
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) AS electric_cars,
    COUNT(sv.uid) AS total_cars,
    (
        COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) * 100.0 /
        NULLIF(COUNT(sv.uid), 0)
    ) AS proportion_electric
FROM 
    spatial_unit AS su
JOIN 
    stock_vehicles AS sv ON su.spatialunit_uid = sv.spatialunit_uid
WHERE 
    sv.year = 2010
    AND su.name = 'Geneva'
GROUP BY 
    su.name
HAVING 
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) > 0
ORDER BY 
    proportion_electric DESC;
```

Note: This query assumes that Geneva is represented by the `spatialunit_uid` 'Geneva' (in the format 'Geneva_ADM3'). If the `spatialunit_uid` is not 'Geneva', you should replace 'Geneva' with the correct `spatialunit_uid`. \n
"""


correction_prompt = """
The answer you gave had the following result. \n
Result: [] \n
Column names: ['city', 'electric_cars', 'total_cars', 'proportion_electric'] \n
Propose a correction to the SQL query and detail the steps of this solution.
"""

chat_response = client.chat.send(
    model=model,
    messages=[
        {"role": "system", "content": cot_intro_prompt},
        {"role": "user", "content": database_info_examples_and_questions + answer + correction_prompt},
    ]
)
print("Chat response:")
print(chat_response.model_dump()["choices"][0]["message"]["content"])

Chat response:
The issue with the provided SQL query is that it is trying to filter the results based on the proportion of electric vehicles. However, since we are joining the `stock_vehicles` table with the `spatial_unit` table, we are getting all the rows from the `spatial_unit` table, not just the rows where there are electric vehicles.

To correct this, we need to filter the `stock_vehicles` table based on the `spatial_unit` table before joining it with the `spatial_unit` table. We also need to filter the results based on the year and city.

Here's the corrected SQL query:

```sql
SELECT 
    su.name AS city,
    COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) AS electric_cars,
    COUNT(sv.uid) AS total_cars,
    (
        COUNT(CASE WHEN sv.fuel_type LIKE '%electric%' THEN sv.uid END) * 100.0 /
        NULLIF(COUNT(sv.uid), 0)
    ) AS proportion_electric
FROM 
    spatial_unit AS su
JOIN 
    stock_vehicles AS sv ON su.spatialunit_uid = sv.spatialunit_uid
WHERE 


#### Statbot-compliant version

In [ ]:
# Install packages
# Carefull requires python 3+ (current is 3.13.8)

%pip install langchain
%pip install -U langchain-community
%pip install sentence-transformers
%pip install chromadb

In [ ]:
# install packages 2

#%pip install -r ../statbot-api/requirements.txt
# Warning too long !!

In [4]:
# Define imports

import pandas as pd
#import os
#import sys
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
#from langchain.vectorstores.chroma import Chroma
from langchain_community.vectorstores import Chroma
#from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

import time
import sys
import os
from sqlalchemyWrapper import (
    schema_db_postgres_statbot_zhaw
)
from langchain_classic.chains import LLMChain
import tiktoken
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI

In [5]:
def few_shot_template_examples(example_prompt, example_selector):
    prefix = '''You are an helpful AI assistant who writes SQL query for a given question. Given the database described by the database schema below, write a SQL query that answers the question.\nDo not explain the SQL query.\nReturn just the query, so it can be run verbatim from your response.\n### Database Schema\n{table_info}
    '''

    few_shot_prompt = FewShotPromptTemplate(
        # These are the examples we want to insert into the prompt.
        example_selector=example_selector,
        example_prompt=example_prompt,
        # The prefix is some text that goes before the examples in the prompt.
        # Usually, this consists of intructions.
        prefix=prefix,
        # The suffix is some text that goes after the examples in the prompt.
        # Usually, this is where the user input will go
        suffix="### Question\n{input}\n### SQL query\n",
        # The input variables are the variables that the overall prompt expects.
        input_variables=["input", "table_info"],
        example_separator="\n\n",
    )
    return few_shot_prompt



def generate_sql_in_context_learning_similar_shots(question, table_name, n_shots=3, file_path="data/query_questions_db.csv"):
    # find the n_shots closest questions from the query_questions_db and the table

    with open(file_path) as f:
        origin_of_shots = pd.read_csv(f, delimiter=',')

    examples = origin_of_shots.loc[origin_of_shots['db_id']==table_name]
    examples = examples.reset_index()
    few_shot_examples = []
    meta_data = []

    for j in range(len(examples)):
        ex_question = examples.loc[j, 'question'].replace("\n", "").strip()
        ex_query = examples.loc[j, 'query']
        few_shot_examples.append({"question": ex_question})
        meta_data.append({"question": ex_question, "query": ex_query})

    to_vectorize = [" ".join(example.values()) for example in few_shot_examples]

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")

    vectorstore = None
    if vectorstore is not None:
        # CLEAR THE VECTORSTORE
        vectorstore.delete_collection()

    vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=meta_data)

    # Lower score is more similar
    answers = vectorstore.similarity_search_with_score(query=question, k=n_shots)

    examples_selector = SemanticSimilarityExampleSelector(
        vectorstore=vectorstore,
        k=n_shots,
    )
    examples_prompt = PromptTemplate(
        input_variables=["question", "query"],
        template="### Question\n{question}\n### SQL query\n{query}",
    )
    prompt_template = few_shot_template_examples(examples_prompt, examples_selector)

    return prompt_template

In [ ]:
def query_engineering_and_call(question, table_name, qry_id):

    sys.stderr.write(f"inside query_engineering_and_call" + "/n")
    
    prompt_template = generate_sql_in_context_learning_similar_shots(question, table_name)

    # For testing purposes
    # prompt_template = zero_shot_template()

    model_name = os.environ["MODEL_NAME"]
    model_path = os.environ["MODEL_PATH"]

    inference_server_url = os.environ["INFERENCE_SERVER_URL"]
    deployed_llm_token = os.environ["DEPLOYED_LLM_TOKEN"]

    llm = ChatOpenAI(
        model=model_path + model_name,
        openai_api_key=deployed_llm_token,
        openai_api_base=inference_server_url,
        max_tokens=1500,
        n=1,
        stream=False,
        top_p=1.0,
        frequency_penalty=0.0,
        presence_penalty=0.0,
        temperature=0.0
    )

    tic = time.perf_counter()
    
    llm_chain = prompt_template | llm
    sql = None

    ddl = schema_db_postgres_statbot_zhaw(include_tables=['spatial_unit', table_name],
                             sample_number=5)

    llm_inputs = {
        "input": question,
        "table_info": ddl,
    }

    sys.stderr.write(f"llm_inputs: {llm_inputs}\n")

    prompt_strings = prompt_template.format(input = question, table_info = ddl)
    sys.stderr.write(f"Prompt_string: {prompt_strings}\n")

    sys.stderr.write(f"Starting  generation:\n")
    while sql is None:
        try:
            # sql = llm_chain.run(**llm_inputs)
            sql = llm_chain.invoke(llm_inputs)
            sys.stderr.write(f"Question: {question}\n")
            sys.stderr.write(f"sql: {sql}\n")
        except Exception as e:
            sys.stderr.write(str(e))
            time.sleep(3)
            pass
    
    # time 
    toc = time.perf_counter()

    num_tokens = sql.response_metadata['token_usage']['total_tokens']
    sql_response = sql.content
    
    process_time = toc-tic
    print(f"Process Time= {process_time:0.4f} second")
    r = {"message": {
        "db_id": table_name,
        "id": qry_id,
        "generated_query": sql_response.replace("\n", " ").replace("\n\n", " ").replace(" ", " ").replace("  ", " "),
        "prompt": prompt_strings,
        "question": question,
        "time": process_time,
        "num_tokens": num_tokens,
    }}

    return r

### Query check (no database verification)

Ask the LLM to check its query using the original prompt and the generated output. No check with the database (yet).

### Query check (database verification)

Ask the LLM to check its query using the original prompt, the generated output and the database answer. Check can depend on database answer (execution error, empty answer, out of scope answer)